# Stock Scout AI — V1

## M1 Data Engine + M2 Technical Engine

Pipeline:

**S&P 500 → Data Engine → Scanner → Technical Engine → Top 10 → Paper Portfolio**

Questa V1 è deliberatamente deterministica e tracciabile. L'Intelligence Agent, il learning loop e le challenger strategies arriveranno dopo la stabilizzazione di M1 + M2.


## 1. Setup Colab

La cella seguente clona sempre la versione corrente della repository GitHub e installa le dipendenze.


In [ ]:
!rm -rf /content/stock-scout-ai
!git clone https://github.com/freshfrisk666-creator/stock-scout-ai.git /content/stock-scout-ai
%cd /content/stock-scout-ai
!python -m pip install --upgrade pip -q
!pip -q install -r requirements.txt


## 2. Import e configurazione


In [ ]:
from pathlib import Path
import pandas as pd

from main import load_config, run_pipeline

cfg = load_config()
print('Repository:', Path.cwd())
print('History period:', cfg['market']['history_period'])
print('Minimum average dollar volume:', cfg['scanner']['min_avg_dollar_volume_20d'])
print('Paper portfolio capital:', cfg['portfolio']['starting_cash'])


## 3. Esecuzione completa della V1

Il comando seguente esegue M1 + M2, costruisce il Top 10 e genera gli ordini del paper portfolio.


In [ ]:
scan, top10, paper = run_pipeline(cfg)

print(f'Scanned symbols passing liquidity filter: {len(scan)}')
print(f'Top 10 rows: {len(top10)}')
print(f'Paper orders: {len(paper)}')


## 4. Top 10 — output principale


In [ ]:
top10_view = top10[[
    'rank', 'ticker', 'technical_score',
    'entry', 'stop', 'target', 'risk_reward'
]].copy()

top10_view.round({
    'technical_score': 2,
    'entry': 2,
    'stop': 2,
    'target': 2,
    'risk_reward': 2,
})


## 5. Metodologia tracciabile

Il punteggio tecnico è composto da:

- Trend: 30%
- Momentum: 25%
- RSI: 15%
- Volume: 15%
- Breakout: 15%

Trade plan V1:

- Entry = ultimo prezzo disponibile
- Stop = Entry − 1.5 × ATR(14)
- Target = Entry + 3 × ATR(14)


In [ ]:
trace_cols = [
    'ticker', 'price', 'sma20', 'sma50', 'sma200', 'rsi14', 'atr14',
    'momentum_21d', 'momentum_63d', 'volume_ratio_20d',
    'breakout_strength_20d', 'trend_score', 'momentum_score',
    'rsi_score', 'volume_score', 'breakout_score',
    'technical_score', 'entry', 'stop', 'target', 'risk_reward',
]

display(top10[trace_cols].round(3))


## 6. Paper Portfolio


In [ ]:
display(paper.round(2))

cash_remaining = float(paper['cash_remaining'].iloc[-1]) if not paper.empty else cfg['portfolio']['starting_cash']
invested = float(paper['notional'].sum()) if not paper.empty else 0.0

print(f'Invested: {invested:,.2f}')
print(f'Cash remaining: {cash_remaining:,.2f}')


## 7. Database

La run salva automaticamente quattro tabelle SQLite:

`scans`, `technical_signals`, `top10`, `paper_orders`.


In [ ]:
from database.db import get_connection

db_path = cfg['database']['path']
with get_connection(db_path) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        conn,
    )

display(tables)
print('SQLite DB:', db_path)


## 8. V1 checkpoint

Il milestone è raggiunto quando questa notebook run produce:

1. universo S&P 500 scaricato;
2. filtro di liquidità;
3. indicatori tecnici calcolati;
4. Top 10 ordinato con score tracciabile;
5. entry / stop / target / risk-reward;
6. paper orders;
7. salvataggio SQLite.

Solo dopo questo checkpoint aggiungeremo Intelligence, news, learning loop, challenger strategies e backtest.
